# Static Solve — Hanging Cubic Box

This notebook builds a complete static equilibrium solve using only
Python-facing APIs and assets from `pypgo/examples/assets`.

The setup mirrors a "box hanging under gravity" scene:

1. Load a cubic `.veg` volume mesh.
2. Build a `SimulationMesh`.
3. Construct a stable Neo-Hookean deformation energy.
4. Convert gravity into a linear potential energy.
5. Fix a small top-corner patch directly in the notebook.
6. Solve the static energy minimization with Newton.
7. Visualize the rest and solved shapes.

This is a **static solve**. It does not use the dynamic time integrator
or contact/IP C lifecycle.


In [1]:
from pathlib import Path

import numpy as np
import pypgo as pgo
import pypgo.energy as pe
import pypgo.fem as pf
import pypgo.solver as ps
from pypgo import vis
from pypgo.mesh.veg import VolumeMesh, read_veg


## 1. Locate the Example Assets

The notebook intentionally uses `pypgo/examples/assets` rather than
any simulator-specific example directory. The fixed vertex set is
derived from the loaded mesh in the next step.


In [2]:
PACKAGE_ROOT = Path(pgo.__file__).resolve().parent
ASSET_DIR = PACKAGE_ROOT / "examples" / "assets"
OUTPUT_DIR = PACKAGE_ROOT / "examples" / "outputs"
CUBIC_BOX = ASSET_DIR / "veg" / "cubic" / "box.veg"
BOX_SURFACE = ASSET_DIR / "obj" / "box.obj"

print("volume asset:", CUBIC_BOX)
print("embedded surface asset:", BOX_SURFACE)
print("output dir:", OUTPUT_DIR)


volume asset: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/assets/veg/cubic/box.veg
embedded surface asset: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/assets/obj/box.obj
output dir: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/outputs


## 2. Load the Cubic Volume Mesh

`read_veg()` returns a `VegFile` payload. `VolumeMesh.from_veg_file()`
turns that payload into a solver-facing volumetric mesh with geometry,
material regions, and mass-matrix support.


In [3]:
veg = read_veg(str(CUBIC_BOX))
volume = VolumeMesh.from_veg_file(veg)
cubic_data = volume.mesh_data

bbox_min, bbox_max = cubic_data.bbox
embedded_surface = pgo.mesh.read_obj(str(BOX_SURFACE))
surface_embedding = pgo.mesh.SurfaceEmbedding(embedded_surface, volume)

x_levels = np.unique(np.round(cubic_data.vertices[:, 0], decimals=12))
z_levels = np.unique(np.round(cubic_data.vertices[:, 2], decimals=12))
corner_patch_mask = (
    np.isclose(cubic_data.vertices[:, 1], bbox_max[1])
    & (cubic_data.vertices[:, 0] <= x_levels[1])
    & (cubic_data.vertices[:, 2] <= z_levels[1])
)
fixed_vertices = np.flatnonzero(corner_patch_mask).astype(np.int64)

print(volume)
print("geometry:", cubic_data.num_vertices, "vertices,", cubic_data.num_elements, "cubes")
print("material:", volume.material)
print("bbox:", cubic_data.bbox)
print("embedded surface:", embedded_surface.num_vertices, "vertices,", embedded_surface.num_elements, "triangles")
print(
    "surface interpolation:",
    surface_embedding.interpolation_matrix.shape,
    "nnz:",
    surface_embedding.interpolation_matrix.nnz,
)
print("fixed top-corner patch vertices:", fixed_vertices.tolist())
print("num fixed vertices:", len(fixed_vertices))

vis.plot_volume_surface(cubic_data, titles=["rest cubic box"], show_edges=True)


4656,582,375
VolumeMesh(type=MeshType.Cubic, vertices=125, elements=64)
geometry: 125 vertices, 64 cubes
material: ENuMaterial(name='defaultMaterial', density=1000.0, E=10000000.0, nu=0.45)
bbox: (array([-0.25    ,  0.378135, -0.25    ]), array([0.25    , 0.878135, 0.25    ]))
embedded surface: 194 vertices, 384 triangles
surface interpolation: (582, 375) nnz: 4656
fixed top-corner patch vertices: [20, 21, 45, 46]
num fixed vertices: 4


Widget(value='<iframe src="http://localhost:55073/index.html?ui=P_0x165b3e5d0_0&reconnect=auto" class="pyvista…

## 3. Build the Simulation Mesh and Deformation Energy

The asset is a cubic/hexahedral mesh, so the formulation is
`LinearCubic()`. The JSON-style `stable-neo` material maps to
`pf.StableNeo()`.


In [4]:
sim_mesh = pgo.sim.SimulationMesh.create_volumetric(volume)

deformation = pf.deformation_energy(
    sim_mesh,
    formulation=pf.LinearCubic(),
    elastic=pf.StableNeo(),
    plastic=pf.VolumetricPlasticity(dofs=6),
)

print("mesh_type:", sim_mesh.mesh_type)
print("vertices:", sim_mesh.num_vertices)
print("elements:", sim_mesh.num_elements)
print("DOFs:", deformation.num_dofs)
print("state_kind:", deformation.state_kind)


[2026-06-03 00:18:24.173] [info] [deformationModelFactory.cpp:57] Building deformation energy with formulation: hex_trilinear
[2026-06-03 00:18:24.173] [info] [deformationModelManager.cpp:262] Initializing element models (manager path)...
[2026-06-03 00:18:24.175] [info] [deformationModelAssembler.cpp:95] Assembler parameter:0,6
mesh_type: cubic
vertices: 125
elements: 64
DOFs: 375
state_kind: displacement


## 4. Convert Gravity into a Linear Potential

Static solve minimizes potential energy. If `f` is the gravity force
vector, the external potential is `-f^T u`, so we pass `-f` to
`LinearEnergy`.

For this compact tutorial we densify the small mass matrix. For larger
meshes, a sparse matvec helper would be preferable.


In [5]:
mass = volume.mass_matrix().to_dense()
gravity_accel = np.array([0.0, -981, 0.0], dtype=np.float64)
gravity_accel_dofs = np.tile(gravity_accel, sim_mesh.num_vertices)

gravity_force = mass @ gravity_accel_dofs
gravity_energy = pe.LinearEnergy(-gravity_force)

print("mass matrix:", mass.shape)
print("gravity force norm:", float(np.linalg.norm(gravity_force)))
print("linear energy DOFs:", gravity_energy.num_dofs)


mass matrix: (375, 375)
gravity force norm: 12545.878360371122
linear energy DOFs: 375


## 5. Convert Fixed Vertices to Fixed DOFs

Each fixed vertex contributes three displacement DOFs: `x`, `y`, and
`z`. Passing `fixed_values=None` means the solver fixes those DOFs to
their current values in `x0`, which is zero displacement here.


In [6]:
fixed_dofs = (3 * fixed_vertices[:, None] + np.arange(3, dtype=np.int64)).ravel()
x0 = np.zeros(deformation.num_dofs, dtype=np.float64)

print("fixed DOFs:", fixed_dofs.tolist())
print("initial fixed values:", x0[fixed_dofs])


fixed DOFs: [60, 61, 62, 63, 64, 65, 135, 136, 137, 138, 139, 140]
initial fixed values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## 6. Compose the Static Objective and Solve

The static objective is:

&&
E(u) = E_\text{elastic}(u) - f_\text{gravity}^T u
&&

The fixed boundary is handled by `solve_newton(..., fixed_dofs=...)`,
not by adding another energy term.


In [7]:
total_energy = pe.EnergySet([
    (deformation, 1.0),
    (gravity_energy, 1.0),
])

result = ps.solve_newton(
    total_energy,
    x0=x0,
    fixed_dofs=fixed_dofs.tolist(),
    fixed_values=None,
    options=ps.NewtonOptions(max_iter=200, tol=1e-4, damping=True, line_search="backtrack"),
)

print("status:", result.status.name)
print("converged:", result.converged)
print("iterations:", result.iterations)
print("final objective:", result.final_objective)
print("final gradient max norm:", result.final_gradient_max_norm)
print("max |u|:", float(np.max(np.abs(result.x))))
print("fixed values after solve:", result.x[fixed_dofs])


[2026-06-03 00:18:33.103] [info] [NewtonSolver.cpp:383] Newton solve time: 0.111086
status: CONVERGED
converged: True
iterations: 19
final objective: -13067.65013635328
final gradient max norm: 0.01298832246004622
max |u|: 0.4420110992541285
fixed values after solve: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## 7. Visualize the Solved Shape

The solver state is a displacement vector. Reshape it to `(n, 3)` and
add it to the rest vertices to build a deformed `CubicMeshData`.
The display surface is driven by `pgo.mesh.SurfaceEmbedding`, which
preserves the `box.obj` surface topology while interpolating volume
displacements.


In [8]:
displacement = result.x.reshape((-1, 3))
deformed_vertices = cubic_data.vertices + displacement
deformed_cubic = pgo.mesh.CubicMeshData(deformed_vertices, cubic_data.elements)
deformed_surface = surface_embedding.deform(result.x)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
deformed_obj = OUTPUT_DIR / "static_solve_box_hang_deformed.obj"
pgo.mesh.write_obj(str(deformed_obj), deformed_surface)

print("rest bbox:    ", cubic_data.bbox)
print("deformed bbox:", deformed_cubic.bbox)
print("deformed surface:", deformed_surface.num_vertices, "vertices,", deformed_surface.num_elements, "triangles")
print("wrote OBJ:", deformed_obj)

vis.plot_surface(
    [embedded_surface, deformed_surface],
    titles=["embedded rest surface", "embedded static solve"],
    colors=["lightgray", "salmon"],
    show_edges=True,
    window_size=(900, 420),
)


Saved mesh (#v: 194, #t: 384) to /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/outputs/static_solve_box_hang_deformed.obj.
rest bbox:     (array([-0.25    ,  0.378135, -0.25    ]), array([0.25    , 0.878135, 0.25    ]))
deformed bbox: (array([-0.49687758,  0.04870181, -0.49687758]), array([0.21819455, 0.878135  , 0.21819455]))
deformed surface: 194 vertices, 384 triangles
wrote OBJ: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/outputs/static_solve_box_hang_deformed.obj


Widget(value='<iframe src="http://localhost:55073/index.html?ui=P_0x162167b90_1&reconnect=auto" class="pyvista…

## 8. Optional: Soft Pin Energy

A JSON-style `coeff` pin is a soft penalty. For a hard static boundary,
prefer `fixed_dofs` as above. If you want a soft attachment term
instead, use `VertexAttachment` and include it in the `EnergySet`.


In [9]:
soft_pin = pe.VertexAttachment(
    sim_mesh=sim_mesh,
    vertex_indices=fixed_vertices,
    target_positions=np.zeros(3 * len(fixed_vertices), dtype=np.float64),
    coeff=1e4,
    is_displacement=True,
)

print("soft pin:", soft_pin)
print("soft pin energy at solved state:", soft_pin.value(result.x))

# add soft pin energy to the total energy and solve again
total_energy_with_soft_pin = pe.EnergySet([
    (deformation, 1.0),
    (gravity_energy, 1.0),
    (soft_pin, 1.0),
])
result_with_soft_pin = ps.solve_newton(
    total_energy_with_soft_pin,
    x0=result.x,
    fixed_dofs=fixed_dofs.tolist(),
    fixed_values=None,
    options=ps.NewtonOptions(max_iter=200, tol=1e-4, damping=True, line_search="backtrack"),
)

print("status:", result_with_soft_pin.status.name)
print("converged:", result_with_soft_pin.converged)
print("iterations:", result_with_soft_pin.iterations)
print("final objective:", result_with_soft_pin.final_objective)
print("final gradient max norm:", result_with_soft_pin.final_gradient_max_norm)
print("max |u|:", float(np.max(np.abs(result_with_soft_pin.x))))
print("soft pin energy at new solved state:", soft_pin.value(result_with_soft_pin.x))

# visualize the new deformed shape
displacement_with_soft_pin = result_with_soft_pin.x.reshape((-1, 3))
deformed_vertices_with_soft_pin = cubic_data.vertices + displacement_with_soft_pin
deformed_cubic_with_soft_pin = pgo.mesh.CubicMeshData(deformed_vertices_with_soft_pin, cubic_data.elements)
deformed_surface_with_soft_pin = surface_embedding.deform(result_with_soft_pin.x)

soft_pin_obj = OUTPUT_DIR / "static_solve_box_hang_soft_pin_deformed.obj"
pgo.mesh.write_obj(str(soft_pin_obj), deformed_surface_with_soft_pin)

print("deformed bbox with soft pin:", deformed_cubic_with_soft_pin.bbox)
print(
    "soft pin deformed surface:",
    deformed_surface_with_soft_pin.num_vertices,
    "vertices,",
    deformed_surface_with_soft_pin.num_elements,
    "triangles",
)
print("wrote soft pin OBJ:", soft_pin_obj)
vis.plot_surface(
    [embedded_surface, deformed_surface_with_soft_pin],
    titles=["embedded rest surface", "embedded soft pin solve"],
    colors=["lightgray", "lightcoral"],
    show_edges=True,
    window_size=(900, 420),
)


soft pin: VertexAttachment(375 DOFs)
soft pin energy at solved state: 0.0
[2026-06-03 00:18:37.525] [info] [NewtonSolver.cpp:383] Newton solve time: 0.042391
status: CONVERGED
converged: True
iterations: 4
final objective: -13067.650136386852
final gradient max norm: 8.393040798182483e-05
max |u|: 0.4420117258582546
soft pin energy at new solved state: 0.0
Saved mesh (#v: 194, #t: 384) to /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/outputs/static_solve_box_hang_soft_pin_deformed.obj.
deformed bbox with soft pin: (array([-0.49687802,  0.04870158, -0.49687802]), array([0.21819446, 0.878135  , 0.21819446]))
soft pin deformed surface: 194 vertices, 384 triangles
wrote soft pin OBJ: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/outputs/static_solve_box_hang_soft_pin_deformed.obj


Widget(value='<iframe src="http://localhost:55073/index.html?ui=P_0x162166d20_2&reconnect=auto" class="pyvista…